# Vein glass

![Closed venation and adaptive flow as leaded glass](../assets/vein_glass_preview.png)

`demo_vein_models` grows vein networks and draws them as glowing lines on a dark leaf. `demo_glass_rose` lights Voronoi cells as glass held up to the sun. This notebook puts the two together: **the veins become the lead, and the areoles they enclose become the panes**, lit with the same glass recipe (facet tilt, glow in the middle of a pane, bevel, hammered texture, bloom) and the same parameter values as the glass rose.

The only change to the glass recipe is where the lead comes from. In the rose, a pixel is lead when it is about as close to its second-nearest seed as to its nearest one (a Voronoi wall). Here it is lead when it lies inside a vein, and a vein's lead is as thick as the vein: Murray widths for Recipe A, conductance for Recipe B. A pane is a connected region of glass between the veins.

That gives a number to check for free. For a vein network drawn in the plane without crossings, Euler's formula says **panes = loops + 1** (the +1 is the outside). A tree has no loops, so it makes only one pane, plus whatever the leaf margin cuts off. See `demo_proximity_graphs` for how Voronoi cells and veins are two views of one structure.

**How to use:** run top to bottom (needs `ipywidgets`).

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as w

from math_vis import finish, glass_lines, make_cmap
from pattern_formation.networks import relax_moving_graph, count_loops
from pattern_formation.venation import (
    BG, GLASS, SHAPES, leaf_outline, grow_veins, leaf_mesh, murray_widths, vein_stats,
    render_vein_glass,
)

FAST = "PYTEST_CURRENT_TEST" in os.environ

plt.rcParams.update({"figure.facecolor": BG, "axes.facecolor": BG, "text.color": "#d9e2dc"})

# the glass rose's lighting (demo_glass_rose), unchanged
ROSE_LIGHT = dict(facet=0.45, glow=0.4, lead_var=0.35, bevel=0.35, texture=0.04, bloom=0.2)
PALETTES = {
    "jewel (stained glass)": (GLASS, (14, 11, 18)),
    "rose": (["#1a0b2e", "#4a1c40", "#b83b5e", "#f08a5d", "#f9ed69"], (14, 11, 18)),
    "backlit leaf": (["#1d2a0c", "#4f6b14", "#93b52a", "#d6e36a", "#fff7c8"], (20, 26, 12)),
}
HEIGHT = 300 if FAST else 900

## 1. Grow a network

The same two recipes and parameter values as `demo_vein_models`:

- **A · open / A · closed**: space colonization, birth distance $b$, kill distance 0.02 (open) or 0.01 (closed), Murray widths with $n = 2$.
- **B · steady / B · moving**: adaptive flow on a leaf mesh of 28 rows, $\gamma = 0.5$, load fluctuation $\alpha$ = 0 or 0.6; vein width $C^{1/4}$ (radius, since $C \propto r^4$), links below $10^{-4} C_\max$ hidden.

In [ ]:
def grow(recipe, shape="ovate", b=0.04, alpha=0.6, seed=0):
    # returns xy, edges, width, cutoff, power: what render_vein_glass needs
    poly = leaf_outline(shape)
    if recipe.startswith("A"):
        closed = recipe.endswith("closed")
        g = grow_veins(poly, b=b, kill=0.01 if closed else 0.02, closed=closed,
                       steps=60 if FAST else 200, seed=seed)
        _, wd = murray_widths(g["parent"], g["edges"])
        return poly, g["xy"], g["edges"], wd, 0.5, 1.0
    xy, edges, base = leaf_mesh(poly, 12 if FAST else 28, seed=seed)
    N = len(xy)
    length = np.linalg.norm(xy[edges[:, 0]] - xy[edges[:, 1]], axis=1)
    sinks = np.array([i for i in range(N) if i != base])
    a = alpha if recipe.endswith("moving") else 0.0
    C = relax_moving_graph(edges, N, base, sinks, a, gamma=0.5, dt=0.5,
                           steps=200 if FAST else 800, length=length)
    return poly, xy, edges, C ** 0.25, (1e-4 * C.max()) ** 0.25, 3.0

## 2. From veins to glass, stage by stage

`flat`: one colour per pane, from the same colour recipe as the stained glass (`cell_colors`, a diagonal trend plus a smooth random field, evaluated at the centre of each pane). `lit`: the glass rose's lighting, with the distance to the nearest vein taking the place of the distance to the seed, so each pane glows in its middle. `leaded`: veins and leaf margin drawn as lead with a bright bevel. `final`: hammered texture and bloom.

In [ ]:
poly, xy, edges, wd, cut, power = grow("A · closed")
img, st = render_vein_glass(xy, edges, wd, poly, height=HEIGHT, cutoff=cut, power=power,
                            stages=True, **ROSE_LIGHT)
fig, axs = plt.subplots(1, 4, figsize=(14, 5.6))
for ax, name in zip(axs, ["flat", "lit", "leaded", "final"]):
    ax.imshow(st[name])
    ax.set_title(name, color="#d9e2dc")
    ax.axis("off")
plt.tight_layout()
plt.show()
print(f"panes {st['panes']}   loops {vein_stats(xy, edges, wd, cut)['loops']}")

**Question 1.** The cell above printed more panes than loops. Where do the extra panes come from?

<details><summary>Answer</summary>

From the leaf margin. It is drawn as lead too, so every vein that reaches the margin closes off a region between itself, the margin and its neighbour. Those faces are loops of the network *veins + margin*, not of the veins alone. The next section removes the margin to check Euler's formula on the veins alone.

</details>

## 3. The window

Pick a recipe and a palette. The lighting is the glass rose's; `lead` sets the thinnest and thickest vein in pixels.

In [ ]:
def window(recipe, shape, b, alpha, palette, lead_max, seed):
    poly, xy, edges, wd, cut, power = grow(recipe, shape, b, alpha, seed)
    pal, lead_color = PALETTES[palette]
    img, st = render_vein_glass(xy, edges, wd, poly, height=HEIGHT, cutoff=cut, power=power,
                                lead=(1.0, lead_max), palette=pal, lead_color=lead_color,
                                seed=seed, stages=True, **ROSE_LIGHT)
    fig, ax = plt.subplots(figsize=(6, 8))
    ax.imshow(finish(img, grain=2.0, display=False))
    ax.axis("off")
    plt.show()
    print(f"panes {st['panes']}   loops {vein_stats(xy, edges, wd, cut)['loops']}")


ctrl = dict(
    recipe=w.Dropdown(options=["A · open", "A · closed", "B · steady", "B · moving"], value="B · moving"),
    shape=w.Dropdown(options=SHAPES, value="ovate"),
    b=w.FloatSlider(value=0.04, min=0.025, max=0.1, step=0.005, readout_format=".3f", description="b (A)"),
    alpha=w.FloatSlider(value=0.6, min=0.0, max=1.0, step=0.1, description="alpha (B)"),
    palette=w.Dropdown(options=list(PALETTES), value="backlit leaf"),
    lead_max=w.FloatSlider(value=9.0, min=3.0, max=16.0, step=1.0, description="lead"),
    seed=w.IntSlider(value=0, min=0, max=10, description="seed"),
)
_ = w.interact_manual(window, **ctrl)
if FAST:
    window("B · moving", "ovate", 0.04, 0.6, "backlit leaf", 9.0, 0)

**Question 2.** Recipe B, raise $\alpha$ from 0 to 1. What happens to the window?

<details><summary>Answer</summary>

At $\alpha = 0$ the steady load gives a tree, so the glass falls apart into a few long strips cut off by veins that reach the margin (15 panes on the ovate leaf). As $\alpha$ grows, loops appear and each loop is a new pane: the window fills with small panes, finest near the edge where the thin veins are. The glass picture is a direct view of the loop count in `demo_vein_models`, Question 4.

</details>

## 4. Number the calculation predicted: panes = loops + 1

Draw the veins only (no margin, no leaf mask, thin lead), count the connected regions of glass, and compare with loops $= E - N + K$ from `vein_stats`. The flow mesh of Recipe B is a Delaunay graph, which is planar, so the prediction should hold exactly. Panes smaller than 10 pixels are slivers where lines meet at sharp angles and are counted as lead.

In [ ]:
def euler_check(recipe, height, **kw):
    poly, xy, edges, wd, cut, _ = grow(recipe, **kw)
    s = height / 1.1
    px = np.column_stack([(xy[:, 0] + 0.55) * s, (1.05 - xy[:, 1]) * s])
    keep = wd > cut
    _, st = glass_lines(px[edges[keep]], 0.8, make_cmap(["#000", "#fff"]), int(1.1 * s), height,
                        texture=0, bloom=0, stages=True)
    return st["panes"], vein_stats(xy, edges, wd, cut)["loops"] + 1


print(f"{'network':40s}{'panes':>7s}{'loops + 1':>11s}")
cases = [("B · moving", dict(alpha=a, seed=0)) for a in (0.3, 0.6, 1.0)]
cases += [("A · closed", dict(b=0.05, seed=s)) for s in (0, 1, 2)]
for recipe, kw in cases[:2] if FAST else cases:
    p, l = euler_check(recipe, 600 if FAST else 1400, **kw)
    print(f"{recipe + ' ' + str(kw):40s}{p:7d}{l:11d}")

**What held.** Recipe B: panes = loops + 1 exactly, at every $\alpha$ (57, 70 and 74 at $\alpha$ = 0.3, 0.6 and 1.0). This is a test in `pattern_formation/tests/test_venation.py`.

**What did not, quite.** Recipe A (closed venation) comes out one or three panes short at 1400 pixels (29 against 32 for seed 0). Its veins never cross (checked), and at 4000 pixels the gap shrinks to at most one, so the missing panes are loops smaller than a pixel or two. They are made by the joining rule of closed venation, which links nearby veins at a source; `demo_vein_models` already flags that rule as mine, not the paper's.

## 5. What did we learn?

- The stained glass and the venation are the same object seen from two sides: the glass shows faces, the veins show links.
- A tree makes no glass. Loops are panes, and counting panes counts loops.
- Everything that changes the loop count in `demo_vein_models` ($\alpha$, $\gamma$, open or closed) changes the window.

**Open:** Do the panes of a loopy vein network look like Voronoi cells? Mean number of sides, Lewis's law (pane area grows linearly with the number of sides) and the angles where veins meet would say whether areoles are closer to a foam, a Voronoi tessellation or a crack pattern.

In [ ]:
# Final render: the four recipes side by side
def gallery(palette):
    pal, lead_color = PALETTES[palette]
    fig, axs = plt.subplots(1, 4, figsize=(16, 6))
    for ax, recipe in zip(axs, ["A · open", "A · closed", "B · steady", "B · moving"]):
        poly, xy, edges, wd, cut, power = grow(recipe)
        img = render_vein_glass(xy, edges, wd, poly, height=HEIGHT, cutoff=cut, power=power,
                                palette=pal, lead_color=lead_color, **ROSE_LIGHT)
        ax.imshow(finish(img, grain=2.0, display=False))
        ax.set_title(recipe, color="#d9e2dc")
        ax.axis("off")
    plt.tight_layout()
    plt.show()


gallery("jewel (stained glass)")